# UK Online Retail Pandas Analysis

## Project Title

UK Online Retail Pandas Analysis

## Business Context

This notebook uses Python and pandas to explore historical UK online retail transaction data. The goal is to clean and validate the dataset, analyse revenue trends, review product and customer contribution, and identify practical business observations.

This notebook is a standalone Python / pandas exploratory data analysis project. It complements, but is separate from, the main UK Retail Power BI / BigQuery dashboard project.

## Dataset Overview

The project uses the Online Retail II dataset, identified in the existing project documentation as sourced from the UCI Machine Learning Repository.

Main fields used in the analysis:

- `Invoice`: invoice number
- `StockCode`: product or transaction code
- `Description`: product description
- `Quantity`: quantity purchased or returned
- `InvoiceDate`: transaction date and time
- `Price`: unit price
- `Customer_ID`: customer identifier
- `Country`: customer country

The raw dataset is expected locally at `data/raw/online_retail_II.xlsx` and is excluded from GitHub.

## Tools Used

- Python
- pandas
- Jupyter Notebook
- matplotlib
- openpyxl for reading Excel files through pandas

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Support running the notebook from either the project root or the notebooks folder.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'online_retail_II.xlsx'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

## Load Data

The workbook contains two yearly sheets. Both sheets are loaded and combined so the analysis can cover the full available period.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}. See data/README.md for setup instructions.'
    )

df_2009_2010 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010')
df_2010_2011 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011')

df_all = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
df_all.head()

## Initial Data Checks

This section checks the dataset structure before cleaning. These checks help identify missing values, duplicate rows, and columns that need type conversion.

In [ ]:
print(f'Rows: {df_all.shape[0]:,}')
print(f'Columns: {df_all.shape[1]:,}')
df_all.info()

In [ ]:
missing_summary = (
    df_all.isna()
    .sum()
    .to_frame('missing_rows')
    .assign(missing_pct=lambda x: x['missing_rows'] / len(df_all) * 100)
    .query('missing_rows > 0')
    .sort_values('missing_rows', ascending=False)
)

missing_summary

In [ ]:
duplicate_rows = df_all.duplicated().sum()
print(f'Duplicate rows: {duplicate_rows:,}')

## Data Cleaning

Cleaning steps used in this notebook:

- standardise column names
- convert invoice dates to datetime
- convert customer IDs to a nullable integer type
- convert stock codes to text
- create a revenue field
- identify cancelled invoices and returns
- create a valid sales dataset for revenue analysis

Cancelled invoices are not removed from the full dataset because they are useful for separate return and cancellation analysis.

In [ ]:
df_all = df_all.copy()
df_all.columns = df_all.columns.str.strip().str.replace(' ', '_')

df_all['InvoiceDate'] = pd.to_datetime(df_all['InvoiceDate'])
df_all['Customer_ID'] = df_all['Customer_ID'].astype('Int64')
df_all['StockCode'] = df_all['StockCode'].astype(str)
df_all['Invoice'] = df_all['Invoice'].astype(str)

df_all['TotalSales'] = df_all['Quantity'] * df_all['Price']
df_all['Is_Return'] = df_all['Invoice'].str.startswith('C')
df_all['YearMonth'] = df_all['InvoiceDate'].dt.to_period('M').astype(str)

# Valid sales exclude cancellations/returns and non-positive quantity or price values.
df_sales = df_all[
    (~df_all['Is_Return'])
    & (df_all['Quantity'] > 0)
    & (df_all['Price'] > 0)
].copy()

df_all.to_csv(PROCESSED_DIR / 'retail_with_returns.csv', index=False)
df_sales.to_csv(PROCESSED_DIR / 'retail_cleaned.csv', index=False)

print(f'Rows including returns/cancellations: {len(df_all):,}')
print(f'Rows used for valid sales analysis: {len(df_sales):,}')

In [ ]:
cleaning_check = pd.DataFrame({
    'metric': [
        'Rows in full dataset',
        'Rows in valid sales dataset',
        'Cancelled or return rows',
        'Rows missing Customer_ID in valid sales',
        'Valid sales revenue'
    ],
    'value': [
        len(df_all),
        len(df_sales),
        int(df_all['Is_Return'].sum()),
        int(df_sales['Customer_ID'].isna().sum()),
        df_sales['TotalSales'].sum()
    ]
})

cleaning_check

## Exploratory Analysis

This section reviews the cleaned sales dataset at a high level before moving into grouped analysis.

In [ ]:
df_sales[['Quantity', 'Price', 'TotalSales']].describe()

In [ ]:
sales_overview = pd.Series({
    'first_invoice_date': df_sales['InvoiceDate'].min(),
    'last_invoice_date': df_sales['InvoiceDate'].max(),
    'invoice_count': df_sales['Invoice'].nunique(),
    'customer_count': df_sales['Customer_ID'].nunique(),
    'product_code_count': df_sales['StockCode'].nunique(),
    'country_count': df_sales['Country'].nunique(),
    'total_revenue': df_sales['TotalSales'].sum()
})

sales_overview

## KPI-Style Analysis

These metrics provide a concise overview of sales activity after cleaning.

In [ ]:
total_revenue = df_sales['TotalSales'].sum()
total_orders = df_sales['Invoice'].nunique()
total_customers = df_sales['Customer_ID'].nunique()
average_order_value = total_revenue / total_orders

kpi_summary = pd.DataFrame({
    'KPI': [
        'Total revenue',
        'Total orders',
        'Identified customers',
        'Average order value'
    ],
    'Value': [
        total_revenue,
        total_orders,
        total_customers,
        average_order_value
    ]
})

kpi_summary

## Revenue Trend Analysis

Revenue is grouped by month to review changes over time and identify possible seasonality.

In [ ]:
monthly_sales = (
    df_sales.groupby('YearMonth', as_index=False)
    .agg(total_revenue=('TotalSales', 'sum'), orders=('Invoice', 'nunique'))
    .sort_values('YearMonth')
)
monthly_sales['average_order_value'] = monthly_sales['total_revenue'] / monthly_sales['orders']

monthly_sales.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly_sales['YearMonth'], monthly_sales['total_revenue'], marker='o')
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
monthly_sales['month_number'] = pd.to_datetime(monthly_sales['YearMonth']).dt.month
seasonal_pattern = (
    monthly_sales.groupby('month_number', as_index=False)['total_revenue']
    .mean()
    .rename(columns={'total_revenue': 'average_monthly_revenue'})
    .sort_values('average_monthly_revenue', ascending=False)
)

seasonal_pattern

## Product Analysis

Products are grouped by stock code and description to identify the highest revenue items. Non-product transaction codes such as postage or manual adjustments are excluded from the filtered product view.

In [ ]:
non_product_codes = ['POST', 'M', 'DOT', 'D', 'BANK CHARGES', 'C2']
df_products = df_sales[~df_sales['StockCode'].isin(non_product_codes)].copy()

top_products = (
    df_products.groupby(['StockCode', 'Description'], dropna=False, as_index=False)
    .agg(total_revenue=('TotalSales', 'sum'), quantity_sold=('Quantity', 'sum'))
    .sort_values('total_revenue', ascending=False)
    .head(10)
)

top_products

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_data = top_products.sort_values('total_revenue')
ax.barh(plot_data['StockCode'], plot_data['total_revenue'])
ax.set_title('Top 10 Product Codes by Revenue')
ax.set_xlabel('Revenue')
ax.set_ylabel('Stock code')
plt.tight_layout()
plt.show()

In [ ]:
product_revenue = df_products.groupby('StockCode')['TotalSales'].sum().sort_values(ascending=False)
product_revenue_share = (product_revenue.head(10).sum() / product_revenue.sum()) * 100
print(f'Top 10 product revenue share: {product_revenue_share:.2f}%')

## Customer Analysis

Customer-level analysis excludes rows where `Customer_ID` is missing. This avoids treating unknown customers as one combined customer group.

In [ ]:
df_customers = df_sales[df_sales['Customer_ID'].notna()].copy()

top_customers = (
    df_customers.groupby('Customer_ID', as_index=False)
    .agg(total_revenue=('TotalSales', 'sum'), orders=('Invoice', 'nunique'))
    .sort_values('total_revenue', ascending=False)
    .head(10)
)

top_customers

In [ ]:
customer_revenue = df_customers.groupby('Customer_ID')['TotalSales'].sum().sort_values(ascending=False)
customer_cumulative_share = customer_revenue.cumsum() / customer_revenue.sum() * 100

top_20_percent_customer_count = int(len(customer_revenue) * 0.2)
top_20_customer_revenue_share = customer_cumulative_share.iloc[top_20_percent_customer_count - 1]

print(f'Total identified customers: {len(customer_revenue):,}')
print(f'Top 20% customer revenue share: {top_20_customer_revenue_share:.2f}%')

## Country Analysis

Country-level analysis shows where revenue is generated geographically.

In [ ]:
top_countries = (
    df_sales.groupby('Country', as_index=False)
    .agg(total_revenue=('TotalSales', 'sum'), orders=('Invoice', 'nunique'))
    .sort_values('total_revenue', ascending=False)
    .head(10)
)

top_countries

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_data = top_countries.sort_values('total_revenue')
ax.barh(plot_data['Country'], plot_data['total_revenue'])
ax.set_title('Top 10 Countries by Revenue')
ax.set_xlabel('Revenue')
ax.set_ylabel('Country')
plt.tight_layout()
plt.show()

## Returns and Cancellations Review

Invoices starting with `C` are treated as cancellations or returns. They are reviewed separately because they can distort valid sales analysis.

In [ ]:
returns = df_all[df_all['Is_Return']].copy()
valid_gross = df_all[(~df_all['Is_Return']) & (df_all['Quantity'] > 0) & (df_all['Price'] > 0)].copy()

monthly_gross = valid_gross.groupby('YearMonth')['TotalSales'].sum()
monthly_returns = returns.groupby('YearMonth')['TotalSales'].sum().abs()
monthly_return_rate = (monthly_returns / monthly_gross * 100).dropna()

monthly_return_summary = pd.DataFrame({
    'gross_sales': monthly_gross,
    'absolute_returns': monthly_returns,
    'return_rate_pct': monthly_return_rate
}).fillna(0)

monthly_return_summary.tail(12)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly_return_summary.index, monthly_return_summary['return_rate_pct'], marker='o')
ax.set_title('Monthly Return Rate')
ax.set_xlabel('Month')
ax.set_ylabel('Return rate (%)')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Findings

These findings are based on a local validation run of this notebook.

- The combined raw dataset contains 1,067,371 rows before filtering.
- After excluding cancellations/returns and non-positive quantity or price values, the valid sales dataset contains 1,041,670 rows.
- The top 10 filtered product codes account for 7.99% of filtered product revenue.
- The top 20% of identified customers account for 77.24% of identified customer revenue.
- Seasonal, country-level, and return-specific observations should be confirmed from the final notebook outputs before publishing.

## Business Recommendations

Based on the analysis areas covered in this notebook:

- Monitor revenue trends by month to identify seasonal changes.
- Review customer concentration risk, especially among high-value customers.
- Identify top products for stock planning and sales planning.
- Investigate returns or cancelled invoices separately from valid sales.
- Use customer, product, and country-level insights to support commercial decision-making.

## Limitations

- The dataset is historical only.
- No marketing cost data is included.
- No product cost or margin data is included.
- No customer demographic data is included.
- Missing customer IDs limit customer-level analysis.
- Returns and cancelled invoices require careful handling.
- Product descriptions may need further standardisation.
- Findings should be validated with additional business context.

## Next Steps

- Add RFM customer segmentation.
- Add cohort analysis.
- Add automated data validation checks.
- Compare pandas results with the Power BI / SQL version.
- Convert repeated analysis steps into reusable Python functions.
- Add more visualisations.